# 🚗 McQueen StyleTTS2 — Inference

**Before running:**
1. Upload `epoch_2nd_00175 (1).pth` as a Kaggle dataset (name it `mcqueen-checkpoint`)
2. Add that dataset as Input on the right sidebar
3. Add your original audio dataset (`asshole`) as Input too (for the reference clip)
4. Set Accelerator to **GPU T4** (needed for fast inference)
5. Hit **Run All**

In [ ]:
# Cell 1 — Install deps (same as trainer, no torch install)
!apt-get update -qq
!apt-get install -y espeak-ng -qq
!git clone https://github.com/yl4579/StyleTTS2.git /kaggle/working/StyleTTS2

!pip install -q pydub munch pyyaml librosa soundfile nltk einops einops-exts phonemizer
!pip install -q 'transformers==4.40.0' --no-deps
!pip install -q 'tokenizers>=0.19,<0.20' huggingface-hub safetensors regex filelock packaging requests
!pip install -q 'cython>=3.0'
!pip install -q git+https://github.com/resemble-ai/monotonic_align.git --no-build-isolation

import torch
print(f'✅ PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Apply patches (same as trainer)
import os, re
os.chdir('/kaggle/working/StyleTTS2')

# Restore losses.py
!curl -s https://raw.githubusercontent.com/yl4579/StyleTTS2/main/losses.py -o losses.py

def patch(path, replacements):
    with open(path, 'r', encoding='utf-8') as f:
        code = f.read()
    for old, new in replacements:
        code = old.sub(new, code) if isinstance(old, re.Pattern) else code.replace(old, new)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(code)

patch('models.py', [
    ("torch.load(model_path, map_location='cpu')",
     "torch.load(model_path, map_location='cpu', weights_only=False)"),
    ("torch.load(model_path)",
     "torch.load(model_path, weights_only=False)"),
])
print('✅ Patches applied')

In [ ]:
# Cell 3 — Download utility models needed for inference
import urllib.request, os
os.makedirs('Utils/ASR', exist_ok=True)
os.makedirs('Utils/JDC', exist_ok=True)

def download(url, path):
    if not os.path.exists(path):
        print(f'Downloading {os.path.basename(path)}...')
        urllib.request.urlretrieve(url, path)
    else:
        print(f'{os.path.basename(path)} — already exists')

download('https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epoch_00080.pth',
         'Utils/ASR/epoch_00080.pth')
download('https://github.com/nickoala/jdc/raw/master/bst.t7',
         'Utils/JDC/bst.t7')
print('✅ Utility models ready')

In [ ]:
# Cell 4 — Load model
import torch, yaml, os
import numpy as np
import torchaudio
import librosa
from munch import Munch

os.chdir('/kaggle/working/StyleTTS2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ⬇️ UPDATE if your dataset name is different
CHECKPOINT = '/kaggle/input/mcqueen-checkpoint/epoch_2nd_00175 (1).pth'

# Build a minimal inference config
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/yl4579/StyleTTS2/main/Configs/config_ft.yml',
    'config_infer.yml'
)
with open('config_infer.yml') as f:
    config = yaml.safe_load(f)

# Load model
from models import *
from utils import *
from text_utils import TextCleaner

textclenaer = TextCleaner()

model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner=None, pitch_extractor=None, bert=None)

# Load checkpoint
params = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
_ = [model[key].load_state_dict(params['net'][key]) for key in model]
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

print(f'✅ Model loaded from {os.path.basename(CHECKPOINT)}')

In [ ]:
# Cell 5 — Set up inference pipeline
import phonemizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

global_phonemizer = phonemizer.backend.EspeakBackend(
    language='en-us', preserve_punctuation=True, with_stress=True
)

from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),
    clamp=False
)

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask+1, lengths.unsqueeze(1))
    return mask

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = preprocess_audio(wave_tensor).to(device)
    return mel_tensor

def compute_style(ref_path):
    wave, sr = librosa.load(ref_path, sr=24000)
    audio, _ = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=24000)
    mel_tensor = preprocess(audio).unsqueeze(1)
    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor)
        ref_p = model.predictor_encoder(mel_tensor)
    return torch.cat([ref_s, ref_p], dim=1)

def inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)

    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)
        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)
        s_pred = sampler(
            noise=torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding=bert_dur,
            embedding_scale=embedding_scale,
            features=ref_s,
            num_steps=diffusion_steps
        ).squeeze(1)
        s = s_pred[:, 128:]
        ref = s_pred[:, :128]
        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)
        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(en, F0_pred, N_pred, ref.squeeze().unsqueeze(0).unsqueeze(0))
    return out.squeeze().cpu().numpy()[..., :-50]

print('✅ Inference pipeline ready!')

In [ ]:
# Cell 6 — 🎙️ GENERATE SPEECH
import soundfile as sf
from IPython.display import Audio, display

# ⬇️ Reference audio — any McQueen clip from your dataset
REF_AUDIO = '/kaggle/input/datasets/infernapeshashank/asshole/Mcqueensample.wav'

# ⬇️ Text to synthesize — go wild!
TEXTS = [
    "Ka-chow! Speed. I am speed.",
    "I'm not the same car I was. I'm better.",
    "Float like a Cadillac, sting like a Beemer.",
]

ref_s = compute_style(REF_AUDIO)

for i, text in enumerate(TEXTS):
    print(f'\n🎙️ Generating: "{text}"')
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10)
    out_path = f'/kaggle/working/mcqueen_output_{i+1}.wav'
    sf.write(out_path, wav, 24000)
    print(f'   Saved: {out_path}')
    display(Audio(wav, rate=24000))

print('\n✅ Done! Download the .wav files from the Output panel.')